In [1]:
   %pip install "psycopg[binary]" pandas

  Using cached psycopg-3.3.6-py3-none-any.whl.metadata (4.4 kB)
Using cached psycopg-3.3.6-py3-none-any.whl (215 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 10.6 MB/s  0:00:000.8 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [psycopg]
Note: you may need to restart the kernel to use updated packages.


In [2]:
   import psycopg
   conn = psycopg.connect(host="localhost", port=5432,
                          dbname="postgres", user="postgres")
   print(conn.info.status)

0


In [3]:
conn.autocommit = True
conn.execute("CREATE DATABASE firmographics")
conn.close()

conn = psycopg.connect(host="localhost", port=5432,
                       dbname="firmographics", user="postgres")
print(conn.info.dbname)  

firmographics


In [5]:
import pandas as pd

def run(sql):
    """Execute a statement; return a DataFrame if it produces rows."""
    with conn.cursor() as cur:
        cur.execute(sql)
        if cur.description:  
            cols = [d.name for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)
    conn.commit()

In [17]:
run("""
drop table if exists company, ceo, city, state, country cascade;

create table country (
    country_id    int,
    country_name  varchar(50) not null unique,
    primary key (country_id)
);

create table state (
    state_id    int,
    state_name  varchar(50) not null,
    country_id  int not null,
    primary key (state_id),
    foreign key (country_id) references country
);

create table city (
    city_id     int,
    city_name   varchar(50) not null,
    state_id    int,
    country_id  int not null,
    primary key (city_id),
    foreign key (state_id) references state,
    foreign key (country_id) references country
);

create table ceo (
    ceo_id          int,
    first_name      varchar(50) not null,
    last_name       varchar(50) not null,
    birth_year      int not null,
    birth_city_id   int,
    birth_state_id  int,
    ceo_start       int not null,
    primary key (ceo_id),
    foreign key (birth_city_id) references city,
    foreign key (birth_state_id) references state,
    check (birth_city_id is not null or birth_state_id is not null),
    check (ceo_start > birth_year)
);

create table company (
    company_id    int,
    company_name  varchar(100) not null unique,
    founded_year  int not null,
    headquarter_id    int not null,
    ceo_id        int not null unique,
    primary key (company_id),
    foreign key (headquarter_id) references city,
    foreign key (ceo_id) references ceo
);
""")

In [39]:
run("""
select table_name
from information_schema.tables
where table_schema = 'public';
""")

,table_name
0,country
1,state
2,city
3,ceo
4,company


In [19]:
run("""
insert into country (country_id, country_name)
values (1, 'USA'), (2, 'India'), (3, 'Japan'), (4, 'China'),
       (5, 'Italy'), (6, 'France'), (7, 'South Korea');

insert into state (state_id, state_name, country_id)
values (1, 'California', 1), (2, 'New York', 1), (3, 'Washington', 1),
       (4, 'Michigan', 1), (5, 'Arkansas', 1), (6, 'Alabama', 1),
       (7, 'Andhra Pradesh', 2), (8, 'Yamaguchi Prefecture', 3),
       (9, 'Zhejiang', 4), (10, 'Lombardy', 5), (11, 'Gyeonggi-do', 7);

insert into city (city_id, city_name, state_id, country_id)
values (1, 'Cupertino', 1, 1), (2, 'Armonk', 2, 1), (3, 'Seattle', 3, 1),
       (4, 'Detroit', 4, 1), (5, 'Bentonville', 5, 1), (6, 'Tokyo', null, 3),
       (7, 'Hangzhou', 9, 4), (8, 'Paris', null, 6), (9, 'Suwon', 11, 7),
       (10, 'Mobile', 6, 1), (11, 'West Godavari', 7, 2), (12, 'Scarsdale', 2, 1),
       (13, 'Royal Oak', 4, 1), (14, 'Milan', 10, 5), (15, 'Seoul', null, 7),
       (16, 'Jacksonville', 5, 1);

insert into ceo (ceo_id, first_name, last_name, birth_year, birth_city_id, birth_state_id, ceo_start)
values (1, 'Tim', 'Cook', 1960, 10, null, 2011),
       (2, 'Arvind', 'Krishna', 1962, 11, null, 2020),
       (3, 'Andy', 'Jassy', 1968, 12, null, 2021),
       (4, 'Mary', 'Barra', 1961, 13, null, 2014),
       (5, 'Hiroki', 'Totoki', 1964, null, 8, 2025),
       (6, 'Eddie', 'Wu', 1975, null, 9, 2023),
       (7, 'Luca', 'de Meo', 1967, 14, null, 2025),
       (8, 'Young-hyun', 'Jun', 1960, 15, null, 2025),
       (9, 'John', 'Furner', 1974, 16, null, 2026);

insert into company (company_id, company_name, founded_year, headquarter_id, ceo_id)
values (1, 'Apple Inc.', 1976, 1, 1), (2, 'IBM', 1911, 2, 2),
       (3, 'Amazon', 1994, 3, 3), (4, 'General Motors', 1908, 4, 4),
       (5, 'Sony Group Corporation', 1946, 6, 5), (6, 'Alibaba Group', 1999, 7, 6),
       (7, 'Kering SA', 1963, 8, 7), (8, 'Samsung Electronics', 1969, 9, 8),
       (9, 'Walmart', 1962, 5, 9);
""")

In [40]:
run("select * from company;")

,company_id,company_name,founded_year,headquarter_id,ceo_id
0,1,Apple Inc.,1976,1,1
1,2,IBM,1911,2,2
2,3,Amazon,1994,3,3
3,4,General Motors,1908,4,4
4,5,Sony Group Corporation,1946,6,5
5,6,Alibaba Group,1999,7,6
6,7,Kering SA,1963,8,7
7,8,Samsung Electronics,1969,9,8
8,9,Walmart,1962,5,9


In [21]:
#1 What are the first names of the CEOs of the companies headquartered in the USA?
run("""
select ce.first_name
from company as cp, ceo as ce, city as ci, country as cn
where cp.ceo_id = ce.ceo_id
  and cp.headquarter_id = ci.city_id
  and ci.country_id = cn.country_id
  and cn.country_name = 'USA';
""")

,first_name
0,Tim
1,Arvind
2,Andy
3,Mary
4,John


In [22]:
#2 Which city hosts the headquarters of the company whose CEO has a last name Totoki?
run("""
select ci.city_name
from company as cp, ceo as ce, city as ci
where cp.ceo_id = ce.ceo_id
  and cp.headquarter_id = ci.city_id
  and ce.last_name = 'Totoki';
""")

,city_name
0,Tokyo


In [23]:
#3 What are the founding years of the companies whose CEOs were born between 1960 and 1969?
run("""
select cp.company_name, cp.founded_year
from company as cp, ceo as ce
where cp.ceo_id = ce.ceo_id
  and ce.birth_year between 1960 and 1969;
""")

,company_name,founded_year
0,Apple Inc.,1976
1,IBM,1911
2,Amazon,1994
3,General Motors,1908
4,Sony Group Corporation,1946
5,Kering SA,1963
6,Samsung Electronics,1969


In [24]:
#4 How long has the CEO of the company founded in 1908 been serving in their position?
run("""
select ce.first_name, ce.last_name,
       2026 - ce.ceo_start as years_ceo
from company as cp, ceo as ce
where cp.ceo_id = ce.ceo_id
  and cp.founded_year = 1908;
""")

,first_name,last_name,years_ceo
0,Mary,Barra,12


In [25]:
#5 What is the age of the CEO of the company based in Hangzhou? 
run("""
select ce.first_name, ce.last_name,
       2026 - ce.birth_year as age
from company as cp, ceo as ce, city as ci
where cp.ceo_id = ce.ceo_id
  and cp.headquarter_id = ci.city_id
  and ci.city_name = 'Hangzhou';
""")

,first_name,last_name,age
0,Eddie,Wu,51


In [41]:
#6 What’s the age of the company whose CEO was born in the state of Tennessee?
run("""
select cp.company_name,
       2026 - cp.founded_year as company_age
from company as cp
inner join ceo as ce on cp.ceo_id = ce.ceo_id
left join city as bc on ce.birth_city_id = bc.city_id
inner join state as bs on (bs.state_id = bc.state_id or bs.state_id = ce.birth_state_id)
where bs.state_name = 'Tennessee';
""")

,company_name,company_age


In [27]:
#7 What are the full names of CEOs of companies founded in the USA before 1965?
run("""
select ce.first_name, ce.last_name, cp.company_name
from company as cp, ceo as ce, city as ci, country as cn
where cp.ceo_id = ce.ceo_id
  and cp.headquarter_id = ci.city_id
  and ci.country_id = cn.country_id
  and cn.country_name = 'USA'
  and cp.founded_year < 1965;
""")

,first_name,last_name,company_name
0,Arvind,Krishna,IBM
1,Mary,Barra,General Motors
2,John,Furner,Walmart


In [28]:
#8 Which companies have CEOs who are younger than 60?
run("""
select cp.company_name
from company as cp, ceo as ce
where cp.ceo_id = ce.ceo_id
  and 2026 - ce.birth_year < 60;
""")

,company_name
0,Amazon
1,Alibaba Group
2,Kering SA
3,Walmart


In [29]:
#9 What is the headquarter city of the company whose CEO was born in India? 
run("""
select ci.city_name
from company as cp
inner join ceo as ce on cp.ceo_id = ce.ceo_id
inner join city as ci on cp.headquarter_id = ci.city_id
left join city as bc on ce.birth_city_id = bc.city_id
left join state as bs on ce.birth_state_id = bs.state_id
inner join country as bn on bn.country_id = coalesce(bc.country_id, bs.country_id)
where bn.country_name = 'India';
""")

,city_name
0,Armonk


In [35]:
#10 Which companies have CEOs who started in their positions after 2014? 
run("""
select cp.company_name, ce.ceo_start
from company as cp, ceo as ce
where cp.ceo_id = ce.ceo_id
  and ce.ceo_start > 2014;
""")

,company_name,ceo_start
0,IBM,2020
1,Amazon,2021
2,Sony Group Corporation,2025
3,Alibaba Group,2023
4,Kering SA,2025
5,Samsung Electronics,2025
6,Walmart,2026


In [42]:
conn.close()